# Vectorization Comparison: Harpy vs SpatialData

Use this notebook in the `harpy` environment.

It compares two vectorization paths on an existing assembled SpatialData store:
- `spatialdata.to_polygons(...)`
- `harpy.shape.vectorize(...)`

The goal is just a quick timing and shape-output sanity check.

In [1]:
from pathlib import Path
from time import perf_counter
import json
import platform
import traceback

import harpy as hp
import pandas as pd
import spatialdata
from spatialdata import read_zarr
from spatialdata.transformations import get_transformation

print("python:", platform.python_version())
print("harpy:", getattr(hp, "__version__", "unknown"))
print("spatialdata:", getattr(spatialdata, "__version__", "unknown"))


python: 3.12.13
harpy: 0.3.7
spatialdata: 0.7.3a0


In [12]:
# Edit these for your run.
STORE_PATH = Path("/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr")
LABEL_LAYER = "nuclear_labels"
SPATIALDATA_OUTPUT_LAYER = "nuclear_boundaries_spatialdata_test"
HARPY_OUTPUT_LAYER = "nuclear_boundaries_harpy_test"

RUN_SPATIALDATA = True
RUN_HARPY = True
SAVE_COMPARE_STORE = True
COMPARE_STORE_PATH = STORE_PATH.with_name(STORE_PATH.stem + "_vectorization_compare.sdata.zarr")


In [3]:
def show(value):
    print(json.dumps(value, indent=2, default=str))


def load_store():
    if not STORE_PATH.exists():
        raise FileNotFoundError(STORE_PATH)
    return read_zarr(STORE_PATH)


def normalize_instance_ids(gdf):
    result = gdf.copy()
    if "label" in result.columns:
        ids = result["label"].astype(int)
    elif "cell_id" in result.columns:
        ids = result["cell_id"].astype(int)
    else:
        ids = pd.Index(result.index).astype(str).astype(int)
    result["instance_id_norm"] = ids
    result = result.sort_values("instance_id_norm").set_index("instance_id_norm", drop=False)
    return result


def summarize_shapes(gdf):
    normalized = normalize_instance_ids(gdf)
    summary = {
        "row_count": int(len(normalized.index)),
        "columns": [str(col) for col in normalized.columns],
        "index_name": str(normalized.index.name),
        "instance_head": [int(x) for x in normalized.index[:5].tolist()],
        "total_area": float(normalized.geometry.area.sum()),
    }
    if len(normalized.index):
        minx, miny, maxx, maxy = normalized.total_bounds
        summary["bounds"] = [float(minx), float(miny), float(maxx), float(maxy)]
    return summary


def compare_shape_outputs(spatialdata_gdf, harpy_gdf):
    sd = normalize_instance_ids(spatialdata_gdf)
    hp_gdf = normalize_instance_ids(harpy_gdf)
    sd_ids = set(int(x) for x in sd.index.tolist())
    hp_ids = set(int(x) for x in hp_gdf.index.tolist())
    common_ids = sorted(sd_ids & hp_ids)
    missing_in_harpy = sorted(sd_ids - hp_ids)
    extra_in_harpy = sorted(hp_ids - sd_ids)

    exact_matches = 0
    area_mismatch_ids = []
    geometry_mismatch_ids = []
    for instance_id in common_ids:
        geom_sd = sd.loc[instance_id].geometry
        geom_hp = hp_gdf.loc[instance_id].geometry
        area_sd = float(geom_sd.area)
        area_hp = float(geom_hp.area)
        if abs(area_sd - area_hp) > 1e-6:
            area_mismatch_ids.append(instance_id)
        if geom_sd.equals(geom_hp):
            exact_matches += 1
        else:
            geometry_mismatch_ids.append(instance_id)

    return {
        "spatialdata_rows": int(len(sd.index)),
        "harpy_rows": int(len(hp_gdf.index)),
        "common_instance_count": int(len(common_ids)),
        "missing_in_harpy_count": int(len(missing_in_harpy)),
        "extra_in_harpy_count": int(len(extra_in_harpy)),
        "missing_in_harpy_head": [int(x) for x in missing_in_harpy[:10]],
        "extra_in_harpy_head": [int(x) for x in extra_in_harpy[:10]],
        "exact_geometry_match_count": int(exact_matches),
        "geometry_mismatch_count": int(len(geometry_mismatch_ids)),
        "geometry_mismatch_head": [int(x) for x in geometry_mismatch_ids[:10]],
        "area_mismatch_count": int(len(area_mismatch_ids)),
        "area_mismatch_head": [int(x) for x in area_mismatch_ids[:10]],
        "total_area_spatialdata": float(sd.geometry.area.sum()),
        "total_area_harpy": float(hp_gdf.geometry.area.sum()),
    }


def attach_shapes_layer(sdata_obj, gdf, output_layer):
    transformations = None
    try:
        existing = get_transformation(gdf, get_all=True)
        if not existing:
            transformations = get_transformation(sdata_obj.labels[LABEL_LAYER], get_all=True)
    except Exception:
        transformations = get_transformation(sdata_obj.labels[LABEL_LAYER], get_all=True)
    return hp.shape.add_shapes_layer(
        sdata_obj,
        input=gdf,
        output_layer=output_layer,
        transformations=transformations,
        overwrite=True,
    )


def maybe_write_compare_store(sdata_obj):
    if not SAVE_COMPARE_STORE:
        print("SAVE_COMPARE_STORE = False")
        return None
    if COMPARE_STORE_PATH.exists():
        import shutil
        shutil.rmtree(COMPARE_STORE_PATH)
    sdata_obj.write(COMPARE_STORE_PATH, overwrite=True)
    return str(COMPARE_STORE_PATH)


In [4]:
sdata = load_store()
print(sdata)
print("labels:", list(sdata.labels.keys()))
print("shapes:", list(sdata.shapes.keys()))
print("tables:", list(sdata.tables.keys()))

no parent found for <ome_zarr.reader.Label object at 0x7efbe411be60>: None
no parent found for <ome_zarr.reader.Label object at 0x7efbe41d7140>: None


SpatialData object, with associated Zarr store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128), (24, 64, 64), (24, 32, 32), (24, 16, 16)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (4895, 4) (2D shapes)
│     ├── 'cell_boundaries_harpy_test': GeoDataFrame shape: (4339, 1) (2D shapes)
│     └── 'nuclear_boundaries': GeoDataFrame shape: (4339, 4) (2D shapes)
└── Tables
      ├── 'agg_cell_labels': AnnData (4895, 24)
      ├── 'agg_nuclear_labels': AnnData (4339, 24)
      └── 'nimbus_table': AnnData (4895, 4)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels), cell_boundaries (Shapes

In [5]:
compare_sdata = load_store()
print("base shapes:", list(compare_sdata.shapes.keys()))
compare_sdata


no parent found for <ome_zarr.reader.Label object at 0x7efbd4493ef0>: None
no parent found for <ome_zarr.reader.Label object at 0x7efbd4251970>: None


base shapes: ['cell_boundaries', 'cell_boundaries_harpy_test', 'nuclear_boundaries']


SpatialData object, with associated Zarr store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128), (24, 64, 64), (24, 32, 32), (24, 16, 16)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (4895, 4) (2D shapes)
│     ├── 'cell_boundaries_harpy_test': GeoDataFrame shape: (4339, 1) (2D shapes)
│     └── 'nuclear_boundaries': GeoDataFrame shape: (4339, 4) (2D shapes)
└── Tables
      ├── 'agg_cell_labels': AnnData (4895, 24)
      ├── 'agg_nuclear_labels': AnnData (4339, 24)
      └── 'nimbus_table': AnnData (4895, 4)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels), cell_boundaries (Shapes

## SpatialData Vectorization

In [6]:
spatialdata_result = None
spatialdata_polygons = None
if RUN_SPATIALDATA:
    try:
        sdata_sd = load_store()
        started = perf_counter()
        spatialdata_polygons = spatialdata.to_polygons(sdata_sd.labels[LABEL_LAYER]).copy()
        elapsed = perf_counter() - started
        spatialdata_result = {
            "seconds": round(elapsed, 3),
            "summary": summarize_shapes(spatialdata_polygons),
        }
        print(f"SpatialData vectorization took {elapsed:.2f}s")
        show(spatialdata_result)
        spatialdata_polygons.head()
    except Exception as exc:
        print(f"SpatialData vectorization failed: {type(exc).__name__}: {exc}")
        traceback.print_exc()
else:
    print("RUN_SPATIALDATA = False")


no parent found for <ome_zarr.reader.Label object at 0x7efbb846b320>: None
no parent found for <ome_zarr.reader.Label object at 0x7efbe41a65d0>: None


SpatialData vectorization took 0.84s
{
  "seconds": 0.844,
  "summary": {
    "row_count": 4339,
    "columns": [
      "label",
      "geometry",
      "instance_id_norm"
    ],
    "index_name": "instance_id_norm",
    "instance_head": [
      1,
      2,
      3,
      4,
      5
    ],
    "total_area": 1985789.5,
    "bounds": [
      -0.5,
      -0.5,
      2047.5,
      2047.5
    ]
  }
}


## Harpy Vectorization

In [7]:
harpy_result = None
harpy_polygons = None
if RUN_HARPY:
    try:
        sdata_hp = load_store()
        if HARPY_OUTPUT_LAYER in sdata_hp.shapes:
            del sdata_hp.shapes[HARPY_OUTPUT_LAYER]
        started = perf_counter()
        sdata_hp = hp.shape.vectorize(
            sdata_hp,
            labels_layer=LABEL_LAYER,
            output_layer=HARPY_OUTPUT_LAYER,
            overwrite=True,
        )
        elapsed = perf_counter() - started
        harpy_polygons = sdata_hp.shapes[HARPY_OUTPUT_LAYER].copy()
        harpy_result = {
            "seconds": round(elapsed, 3),
            "summary": summarize_shapes(harpy_polygons),
        }
        print(f"Harpy vectorization took {elapsed:.2f}s")
        show(harpy_result)
        harpy_polygons.head()
    except Exception as exc:
        print(f"Harpy vectorization failed: {type(exc).__name__}: {exc}")
        traceback.print_exc()
else:
    print("RUN_HARPY = False")


no parent found for <ome_zarr.reader.Label object at 0x7efb94d284a0>: None
no parent found for <ome_zarr.reader.Label object at 0x7efb94d2c500>: None
2026-04-18 12:20:34.429 | INFO     | harpy.shape._manager:_mask_to_polygons_rasterio:339 - Finished vectorizing. Dissolving shapes at the border of the chunks. This can take a couple minutes if input mask contains many chunks.
2026-04-18 12:20:34.486 | INFO     | harpy.shape._manager:_mask_to_polygons_rasterio:345 - Dissolve is done.


Harpy vectorization took 0.73s
{
  "seconds": 0.732,
  "summary": {
    "row_count": 4339,
    "columns": [
      "geometry",
      "instance_id_norm"
    ],
    "index_name": "instance_id_norm",
    "instance_head": [
      1,
      2,
      3,
      4,
      5
    ],
    "total_area": 1987966.0,
    "bounds": [
      0.0,
      0.0,
      2048.0,
      2048.0
    ]
  }
}


## Compare

In [8]:
compare_sdata = load_store()
if spatialdata_polygons is not None:
    compare_sdata = attach_shapes_layer(compare_sdata, spatialdata_polygons, SPATIALDATA_OUTPUT_LAYER)
if harpy_polygons is not None:
    compare_sdata = attach_shapes_layer(compare_sdata, harpy_polygons, HARPY_OUTPUT_LAYER)

print("compare shapes:", list(compare_sdata.shapes.keys()))
compare_sdata


no parent found for <ome_zarr.reader.Label object at 0x7efb9545cd70>: None
no parent found for <ome_zarr.reader.Label object at 0x7efb952e5eb0>: None
2026-04-18 12:20:37.229 | WARNING  | harpy.utils._io:_incremental_io_on_disk:86 - layer with name 'nuclear_boundaries_harpy_test' already exists. Overwriting...


compare shapes: ['cell_boundaries', 'cell_boundaries_harpy_test', 'nuclear_boundaries', 'nuclear_boundaries_spatialdata_test', 'nuclear_boundaries_harpy_test']


SpatialData object, with associated Zarr store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128), (24, 64, 64), (24, 32, 32), (24, 16, 16)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (4895, 4) (2D shapes)
│     ├── 'cell_boundaries_harpy_test': GeoDataFrame shape: (4339, 1) (2D shapes)
│     ├── 'nuclear_boundaries': GeoDataFrame shape: (4339, 4) (2D shapes)
│     ├── 'nuclear_boundaries_harpy_test': GeoDataFrame shape: (4339, 1) (2D shapes)
│     └── 'nuclear_boundaries_spatialdata_test': GeoDataFrame shape: (4339, 2) (2D shapes)
└── Tables
      ├── 'agg_cell_labels': AnnData (4895, 24)
      ├── 'agg_nuclear_labels': AnnData (4339, 24)
      └── 'nimbus_table

In [13]:
written_compare_store = maybe_write_compare_store(compare_sdata)
print("written_compare_store:", written_compare_store)


written_compare_store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata_vectorization_compare.sdata.zarr


In [10]:
polygon_comparison = None
if spatialdata_polygons is not None and harpy_polygons is not None:
    polygon_comparison = compare_shape_outputs(spatialdata_polygons, harpy_polygons)

comparison = {
    "label_layer": LABEL_LAYER,
    "store_path": str(STORE_PATH),
    "spatialdata_output_layer": SPATIALDATA_OUTPUT_LAYER,
    "harpy_output_layer": HARPY_OUTPUT_LAYER,
    "spatialdata": spatialdata_result,
    "harpy": harpy_result,
    "polygon_comparison": polygon_comparison,
    "written_compare_store": written_compare_store,
}
show(comparison)


{
  "label_layer": "nuclear_labels",
  "store_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr",
  "spatialdata_output_layer": "nuclear_boundaries_spatialdata_test",
  "harpy_output_layer": "nuclear_boundaries_harpy_test",
  "spatialdata": {
    "seconds": 0.844,
    "summary": {
      "row_count": 4339,
      "columns": [
        "label",
        "geometry",
        "instance_id_norm"
      ],
      "index_name": "instance_id_norm",
      "instance_head": [
        1,
        2,
        3,
        4,
        5
      ],
      "total_area": 1985789.5,
      "bounds": [
        -0.5,
        -0.5,
        2047.5,
        2047.5
      ]
    }
  },
  "harpy": {
    "seconds": 0.732,
    "summary": {
      "row_count": 4339,
      "columns": [
        "geometry",
        "instance_id_norm"
      ],
      "index_name": "instance_id_norm",
      "instance_head": [
        1,
        2,
        3,
        4,
        5
    

In [11]:
from napari_spatialdata import Interactive

interactive = Interactive(compare_sdata)
interactive.run()


2026-04-18 12:20:53.516 | WARNING  | napari_spatialdata._viewer:__init__:56 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.
2026-04-18 12:21:09.677 | INFO     | napari_spatialdata._viewer:get_sdata_shapes:605 - Multipolygons are present in the data. Only the largest polygon per cell is retained.
2026-04-18 12:21:11.012 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-04-18 12:21:14.541 | INFO     | napari_spatialdata._viewer:get_sdata_shapes:605 - Multipolygons are present in the data. Only the largest polygon per cell is retained.
2026-04-18 12:21:15.694 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-04-18 12:21:15.701 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-04-18 12:21:15.706 | DEBUG    | napari_spatialdata._view:_on_layer_update:56